# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a Croissant-structured dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using [Croissant](https://mlcommons.org/croissant/) schema and accessed via:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and examine the overall Croissant metadata structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset and its metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}\nPublished: {getattr(metadata, 'datePublished', None)}\nVersion: {getattr(metadata, 'version', None)}")
# Print available attributes in metadata
print("\nAvailable metadata attributes:")
print([attr for attr in dir(metadata) if not attr.startswith('_')])

## 2. Data Overview
List all available record sets, their `@id`s, and associated fields/columns.

**Note:** All IDs are referenced by their `@id` field as per Croissant schema best practices.

In [ ]:
# List RecordSets and their fields by Croissant @id
def print_record_sets(ds):
    if not hasattr(ds.metadata, 'record_sets'):
        print("No 'record_sets' found in metadata (Croissant >=1.0.0 uses 'record_sets')")
        return
    record_sets = ds.metadata.record_sets
    for record_set in record_sets:
        print(f"\nRecordSet @id: {getattr(record_set, '@id', None)}")
        print(f"  Name: {getattr(record_set, 'name', None)}")
        print(f"  Description: {getattr(record_set, 'description', None)}")
        if hasattr(record_set, 'fields'):
            print("  Fields and their @id's:")
            for field in getattr(record_set, 'fields', []):
                print(f"    Field @id: {getattr(field, '@id', None)}, name: {getattr(field, 'name', None)}")
        if hasattr(record_set, 'columns'):
            print("  Columns and their @id's:")
            for column in getattr(record_set, 'columns', []):
                print(f"    Column @id: {getattr(column, '@id', None)}, name: {getattr(column, 'name', None)}")

print_record_sets(dataset)

# For convenience in later steps, collect record set @ids
all_recordset_ids = []
if hasattr(dataset.metadata, 'record_sets'):
    for record_set in dataset.metadata.record_sets:
        all_recordset_ids.append(getattr(record_set, '@id', None))
print(f"\nFound record set @id's: {all_recordset_ids}")

## 3. Data Extraction
Load data from a record set into a DataFrame using the record set and field `@id`s.

For this dataset, there may be one or more record sets (tables). We'll demonstrate extraction for every available record set.

In [ ]:
# Extract data for each record set into a pandas DataFrame, referenced by @id
dataframes = {}
for record_set_id in all_recordset_ids:
    print(f"Loading data for RecordSet @id={record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print('  No records found.')

# For further steps, select the first non-empty record set
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break
if main_record_set_id is None:
    print("No record sets with records were found!")
else:
    print(f"\nUsing main_record_set_id = {main_record_set_id}")
    print(f"Available columns: {dataframes[main_record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter based on a numeric field, normalize, group and aggregate. You must use field `@id` values to reference columns.

We'll try a column commonly present in this kind of dataset, such as 'abundance', 'percent_coverage', 'MW', or similar. Adapt to the true numeric column in the dataset.

In [ ]:
# Guess a numeric field for this record set
import numpy as np

df = dataframes[main_record_set_id]

# Heuristic to find a likely numeric column
numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dropna().infer_objects().dtype, np.number)]
if not numeric_cols:
    # Try to coerce floats
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dropna().infer_objects().dtype, np.number)]

print(f"Numeric columns detected: {numeric_cols}")

# Use the first detected numeric field for demo, or skip if none.
if not numeric_cols:
    print("No numeric columns found for EDA.")
else:
    numeric_field_id = numeric_cols[0]
    # Try to filter and normalize
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
    try:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    except Exception as e:
        print(f"Could not filter/normalize: {e}")

    # Try grouping by a likely categorical field
    categorical_cols = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    group_field = None
    if categorical_cols:
        group_field = categorical_cols[0]
        try:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
        except Exception as e:
            print(f"Could not group: {e}")

## 5. Visualization
Visualize the numeric field distribution and, if grouping succeeded, compare across groups.

_This section uses `matplotlib` and/or `seaborn` for plotting. Install as needed._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_cols:
    print('No numeric field to plot.')
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f'Average {numeric_field_id} per {group_field}')
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, reviewing, and analyzing records from a Croissant-structured dataset with `mlcroissant`. Key steps included:
- Listing record sets and fields by their Croissant `@id`
- Loading specific record sets into pandas DataFrames
- Simple EDA, normalization, and grouping by key categorical fields
- Visualizing dataset distributions

Further work:
- Investigate fields and record sets for deeper domain-specific analytics
- Combine this data with external protein or sample metadata
- Build ML models using normalized columns as features.